In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

VOLUME_PATH = "/Volumes/workspace/default/nyc_data_vol"

train_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{VOLUME_PATH}/silver_cmapss_train.csv")
test_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{VOLUME_PATH}/silver_cmapss_test.csv")
rul_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{VOLUME_PATH}/silver_cmapss_rul.csv")

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())
print("RUL rows:", rul_df.count())
train_df.printSchema()

In [0]:
display(train_df.limit(5))
display(test_df.limit(5))
display(rul_df.limit(5))

## 1. Dataset Overview

In [0]:
print("Training columns:", len(train_df.columns))
print("Test columns:", len(test_df.columns))
print("Training engines:", train_df.select("unit_number").distinct().count())
print("Test engines:", test_df.select("unit_number").distinct().count())

print("Training missing values:")
train_df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in train_df.columns]).show()
print("Training duplicates:", train_df.count() - train_df.dropDuplicates().count())

## 2. RUL Distribution

In [0]:
train_df.select("RUL").summary().show()
display(train_df.select("RUL").orderBy("RUL"))

## 3. Engine Lifecycle Analytics

In [0]:
engine_life = (
    train_df.groupBy("unit_number")
    .agg(F.max("cycle").alias("lifetime_cycles"), F.count("*").alias("observations"))
    .orderBy("unit_number")
)
display(engine_life)
engine_life.select("lifetime_cycles", "observations").summary().show()

## 4. RUL Across Engine Lifecycles

In [0]:
selected_engines = [1, 10, 20, 50]
available_engines = [r["unit_number"] for r in train_df.select("unit_number").distinct().orderBy("unit_number").collect()]
selected_engines = [e for e in selected_engines if e in available_engines]

rul_progression = (
    train_df.filter(F.col("unit_number").isin(selected_engines))
    .select("unit_number", "cycle", "RUL")
    .orderBy("unit_number", "cycle")
)
display(rul_progression)

## 5. Sensor Correlation with RUL

In [0]:
sensor_columns = [c for c in train_df.columns if c.startswith("sensor_")]
correlations = [(sensor, train_df.stat.corr(sensor, "RUL")) for sensor in sensor_columns]
correlation_df = spark.createDataFrame(correlations, ["sensor", "correlation_with_RUL"])
display(correlation_df.orderBy(F.abs(F.col("correlation_with_RUL")).desc()))

## 6. Sensor Variability

In [0]:
sensor_stats = []
for sensor in sensor_columns:
    row = train_df.select(
        F.mean(sensor).alias("mean"),
        F.stddev(sensor).alias("stddev"),
        F.min(sensor).alias("min"),
        F.max(sensor).alias("max")
    ).collect()[0]
    sensor_stats.append((sensor, row["mean"], row["stddev"], row["min"], row["max"]))

sensor_stats_df = spark.createDataFrame(sensor_stats, ["sensor", "mean", "stddev", "min", "max"])
display(sensor_stats_df.orderBy(F.col("stddev").desc()))

## 7. Prepare ML Features

In [0]:
operating_columns = ["op_setting_1", "op_setting_2", "op_setting_3"]
feature_columns = ["cycle"] + operating_columns + sensor_columns
target_column = "RUL"

print("Number of features:", len(feature_columns))
print("Features:", feature_columns)

## 8. Engine-Level Train / Validation Split
Do not use `sklearn.train_test_split`. PySpark DataFrames are not array-like inputs for that function, and a row-level random split would also leak information between observations from the same engine.

In [0]:
import random

engine_ids = [
    row["unit_number"]
    for row in train_df.select("unit_number").distinct().orderBy("unit_number").collect()
]

random.seed(42)
random.shuffle(engine_ids)

split_index = int(len(engine_ids) * 0.8)
train_engine_ids = engine_ids[:split_index]
validation_engine_ids = engine_ids[split_index:]

ml_train = train_df.filter(F.col("unit_number").isin(train_engine_ids))
ml_val = train_df.filter(F.col("unit_number").isin(validation_engine_ids))

print("Training engines:", len(train_engine_ids))
print("Validation engines:", len(validation_engine_ids))
print("Training rows:", ml_train.count())
print("Validation rows:", ml_val.count())

## 9. Vectorize Features

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
train_vector = assembler.transform(ml_train)
val_vector = assembler.transform(ml_val)

display(train_vector.select("unit_number", "cycle", "RUL", "features").limit(5))

## 10. Regression Evaluators

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

rmse_evaluator = RegressionEvaluator(labelCol="RUL", predictionCol="prediction", metricName="rmse")
mae_evaluator = RegressionEvaluator(labelCol="RUL", predictionCol="prediction", metricName="mae")
r2_evaluator = RegressionEvaluator(labelCol="RUL", predictionCol="prediction", metricName="r2")

## 11. Linear Regression Baseline

In [0]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol="features", labelCol="RUL")
lr_model = lr.fit(train_vector)
lr_predictions = lr_model.transform(val_vector)

lr_rmse = rmse_evaluator.evaluate(lr_predictions)
lr_mae = mae_evaluator.evaluate(lr_predictions)
lr_r2 = r2_evaluator.evaluate(lr_predictions)

print("Linear Regression")
print("RMSE:", lr_rmse)
print("MAE:", lr_mae)
print("R²:", lr_r2)

## 12. Random Forest Regression

In [0]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="features", labelCol="RUL",
    numTrees=100, maxDepth=10, seed=42
)
rf_model = rf.fit(train_vector)
rf_predictions = rf_model.transform(val_vector)

rf_rmse = rmse_evaluator.evaluate(rf_predictions)
rf_mae = mae_evaluator.evaluate(rf_predictions)
rf_r2 = r2_evaluator.evaluate(rf_predictions)

print("Random Forest")
print("RMSE:", rf_rmse)
print("MAE:", rf_mae)
print("R²:", rf_r2)

## 13. Gradient Boosted Trees

In [0]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(
    featuresCol="features", labelCol="RUL",
    maxIter=50, maxDepth=5, seed=42
)
gbt_model = gbt.fit(train_vector)
gbt_predictions = gbt_model.transform(val_vector)

gbt_rmse = rmse_evaluator.evaluate(gbt_predictions)
gbt_mae = mae_evaluator.evaluate(gbt_predictions)
gbt_r2 = r2_evaluator.evaluate(gbt_predictions)

print("Gradient Boosting")
print("RMSE:", gbt_rmse)
print("MAE:", gbt_mae)
print("R²:", gbt_r2)

## 14. Model Comparison

In [0]:
model_results = spark.createDataFrame(
    [
        ("Linear Regression", float(lr_rmse), float(lr_mae), float(lr_r2)),
        ("Random Forest", float(rf_rmse), float(rf_mae), float(rf_r2)),
        ("Gradient Boosting", float(gbt_rmse), float(gbt_mae), float(gbt_r2))
    ],
    ["model", "RMSE", "MAE", "R2"]
)
display(model_results.orderBy("RMSE"))

best_model_row = model_results.orderBy("RMSE").first()
best_model_name = best_model_row["model"]
print("Selected model:", best_model_name)
print("Validation RMSE:", best_model_row["RMSE"])

## 15. Random Forest Feature Importance

In [0]:
rf_importance = rf_model.featureImportances
importance_data = [(feature_columns[i], float(rf_importance[i])) for i in range(len(feature_columns))]
feature_importance = spark.createDataFrame(importance_data, ["feature", "importance"])
display(feature_importance.orderBy(F.col("importance").desc()))

## 16. Reconstruct Official Test RUL
The RUL file contains the remaining RUL at the final observed test cycle for each engine. For any earlier cycle:

`RUL = terminal_RUL + (last_observed_cycle - current_cycle)`

In [0]:
rul_df = rul_df.withColumn("RUL", F.col("RUL").cast("double"))

test_last_cycle = test_df.groupBy("unit_number").agg(F.max("cycle").alias("last_cycle"))

# FD001 RUL file rows correspond to engine IDs 1..N in order.
rul_with_id = (
    rul_df
    .withColumn("unit_number", F.row_number().over(Window.orderBy(F.monotonically_increasing_id())))
    .withColumnRenamed("RUL", "terminal_RUL")
)

test_terminal_truth = test_last_cycle.join(rul_with_id, "unit_number", "inner")
print("Test engines with RUL:", test_terminal_truth.count())
display(test_terminal_truth.orderBy("unit_number"))

In [0]:
test_truth_df = (
    test_df
    .join(test_terminal_truth.select("unit_number", "last_cycle", "terminal_RUL"), "unit_number", "inner")
    .withColumn("RUL", F.col("terminal_RUL") + (F.col("last_cycle") - F.col("cycle")))
)

print("Test rows with reconstructed RUL:", test_truth_df.count())
display(test_truth_df.select("unit_number", "cycle", "RUL").limit(10))

## 17. Retrain Models on All Labeled Training Data and Predict Test Data

In [0]:
all_train_vector = assembler.transform(train_df)
test_vector = assembler.transform(test_df)

lr_final = lr.fit(all_train_vector)
rf_final = rf.fit(all_train_vector)
gbt_final = gbt.fit(all_train_vector)

lr_test_predictions = lr_final.transform(test_vector).select(
    "unit_number", "cycle", F.col("prediction").alias("lr_prediction")
)
rf_test_predictions = rf_final.transform(test_vector).select(
    "unit_number", "cycle", F.col("prediction").alias("rf_prediction")
)
gbt_test_predictions = gbt_final.transform(test_vector).select(
    "unit_number", "cycle", F.col("prediction").alias("gbt_prediction")
)

test_predictions = (
    test_df.select("unit_number", "cycle")
    .join(lr_test_predictions, ["unit_number", "cycle"])
    .join(rf_test_predictions, ["unit_number", "cycle"])
    .join(gbt_test_predictions, ["unit_number", "cycle"])
)

if best_model_name == "Linear Regression":
    selected_prediction_column = "lr_prediction"
elif best_model_name == "Random Forest":
    selected_prediction_column = "rf_prediction"
else:
    selected_prediction_column = "gbt_prediction"

test_predictions = (
    test_predictions
    .withColumn("selected_model", F.lit(best_model_name))
    .withColumn("final_prediction", F.col(selected_prediction_column))
)

display(test_predictions.limit(10))

## 18. Final Test-Engine Predictions

In [0]:
terminal_window = Window.partitionBy("unit_number").orderBy(F.col("cycle").desc())

terminal_predictions = (
    test_predictions
    .withColumn("rn", F.row_number().over(terminal_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .join(test_terminal_truth.select("unit_number", "terminal_RUL"), "unit_number", "inner")
)

display(
    terminal_predictions.select(
        "unit_number", "cycle", "terminal_RUL",
        "lr_prediction", "rf_prediction", "gbt_prediction",
        "selected_model", "final_prediction"
    ).orderBy("unit_number")
)

## 19. C-MAPSS Asymmetric Score

In [0]:
score_df = (
    terminal_predictions
    .withColumn("error", F.col("final_prediction") - F.col("terminal_RUL"))
    .withColumn(
        "score",
        F.when(F.col("error") < 0,
               F.exp(-F.col("error") / 13.0) - 1.0)
         .otherwise(F.exp(F.col("error") / 10.0) - 1.0)
    )
)

cmaps_score = score_df.agg(F.sum("score").alias("C_MAPSS_score")).first()["C_MAPSS_score"]
print("C-MAPSS score:", cmaps_score)

display(score_df.select(
    "unit_number", "terminal_RUL", "final_prediction", "error", "score"
).orderBy("unit_number"))

## 20. Final Test Metrics at Terminal Cycle

In [0]:
terminal_metrics = terminal_predictions.select(
    "terminal_RUL", "lr_prediction", "rf_prediction",
    "gbt_prediction", "final_prediction"
)

for model_name, prediction_col in [
    ("Linear Regression", "lr_prediction"),
    ("Random Forest", "rf_prediction"),
    ("Gradient Boosting", "gbt_prediction")
]:
    e_rmse = RegressionEvaluator(labelCol="terminal_RUL", predictionCol=prediction_col, metricName="rmse")
    e_mae = RegressionEvaluator(labelCol="terminal_RUL", predictionCol=prediction_col, metricName="mae")
    e_r2 = RegressionEvaluator(labelCol="terminal_RUL", predictionCol=prediction_col, metricName="r2")
    print(
        f"{model_name}: "
        f"RMSE={e_rmse.evaluate(terminal_metrics):.4f}, "
        f"MAE={e_mae.evaluate(terminal_metrics):.4f}, "
        f"R²={e_r2.evaluate(terminal_metrics):.4f}"
    )

## 21. Save Outputs for Visualization

In [0]:
# These tables are created from the results of this notebook.
# If your workspace uses a different writable catalog/schema, change the names below.

model_results.write.mode("overwrite").format("delta").saveAsTable(
    "workspace.default.ml_model_results"
)

feature_importance.write.mode("overwrite").format("delta").saveAsTable(
    "workspace.default.ml_feature_importance"
)

validation_predictions = (
    lr_predictions.select("unit_number", "cycle", "RUL", F.col("prediction").alias("lr_prediction"))
    .join(
        rf_predictions.select("unit_number", "cycle", F.col("prediction").alias("rf_prediction")),
        ["unit_number", "cycle"]
    )
    .join(
        gbt_predictions.select("unit_number", "cycle", F.col("prediction").alias("gbt_prediction")),
        ["unit_number", "cycle"]
    )
)

validation_predictions.write.mode("overwrite").format("delta").saveAsTable(
    "workspace.default.ml_validation_predictions"
)

terminal_predictions.write.mode("overwrite").format("delta").saveAsTable(
    "workspace.default.ml_test_predictions"
)

print("Output tables saved successfully.")

In [0]:
tables = [
    "ml_model_results",
    "ml_feature_importance",
    "ml_validation_predictions",
    "ml_test_predictions"
]

for table in tables:
    (
        spark.table(f"workspace.default.{table}")
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", True)
        .csv(f"/Volumes/workspace/default/nyc_data_vol/exports/{table}")
    )

print("All 4 tables exported as CSV successfully.")